# Cooling-system optimization: from thermal model to quantum execution

This workbench walkthrough expands the pinned FlagQuantum example. Run cells in order; the core quantum definitions, synthesis calls and executions are visible below, rather than hidden behind `workflow.run()`.

**Two separate experiments:** first solve a four-node thermal system with a degree-11 QSVT inverse and compare it with a classical solve. Then evaluate an eight-qubit, one-phase-bit **mock** design circuit using 64 supplied costs and two angle pairs. The mock costs are not freshly computed thermal objectives. This is neither a completed physical design optimizer nor a quantum advantage demonstration.

The optional full design has 20 qubits and five phase bits; it is defined at the end but is not constructed or executed by Run all. Original companion modules remain available through **How this notebook uses FlagQuantum**.

## 1. Problem, data and classical reference

Four nodes exchange heat through six optional links. A six-bit word chooses which links exist. The linear system is `A · temperature_offset = heat`; offsets are in kelvin relative to 293 K ambient. This is a classical physical model, before quantum encoding.

In [ ]:
import json
import math
from dataclasses import dataclass, replace
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, prepare_amplitudes, qfunc
from flagquantum.compiler import ResourceEstimate, estimate_resources
from flagquantum.algorithms.primitives.qft import qft

# Classical numerical phase fitting remains a companion helper, not a quantum execution.
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview.phases import PhaseFit, fit, response

REAL, COMPLEX = torch.float64, torch.complex128
AMBIENT, R_ENV = 293, 0.001
HEAT = (2000.0, 4000.0, -200.0, -2000.0)
EDGES = ((0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3))
RESISTANCES = (0.005, 0.006, 0.006, 0.007, 0.007, 0.008)
CONFIGURATIONS = 1 << len(EDGES)
def build_matrix(word: int = 63) -> torch.Tensor:
    """Regenerate one thermal matrix by its six conductance toggles."""
    if not 0 <= word < CONFIGURATIONS:
        raise ValueError("Expected a six-bit cooling configuration")
    matrix = torch.eye(4, dtype=REAL) / R_ENV
    for index, ((i, j), resistance) in enumerate(zip(EDGES, RESISTANCES, strict=True)):
        if (word >> (len(EDGES) - 1 - index)) & 1:
            value = 1 / resistance
            matrix[i, i] += value
            matrix[j, j] += value
            matrix[i, j] -= value
            matrix[j, i] -= value
    return matrix


def weights() -> tuple[float, ...]:
    return (*tuple(1 / value for value in RESISTANCES), 1 / (2 * R_ENV), 0.0)


def alpha() -> float:
    return 2 * math.fsum(weights())

In [ ]:
matrix = build_matrix(63)
heat = torch.tensor(HEAT, dtype=REAL)
reference = torch.linalg.solve(matrix, heat)
print("All-links thermal matrix:\n", matrix)
print("Classical signed temperature offsets (K):", reference.tolist())
print("Absolute temperatures (K):", (reference + AMBIENT).tolist())
assert torch.allclose(matrix @ reference, heat)

## 2. High-level state preparation → synthesis → executable circuit

FlagQuantum's `@qfunc` describes amplitude preparation on an output register. `synth.synthesize()` compiles it under explicit qubit/workspace limits; `Circuit.from_ir()` brings that compiled program into the larger circuit. The thermal oracle below is built with `fq.Circuit` and explicit matrices; it is **not** an end-to-end high-level synthesis of the thermal problem.

In [ ]:
@qfunc
def prepare(amplitudes: list[float], q: Output[QArray[QBit]]) -> None:
    prepare_amplitudes(amplitudes, 0, q)


def preparation(values: torch.Tensor) -> fq.Circuit:
    data = tuple(float(value) for value in values)
    n = len(data).bit_length() - 1
    program = synth.synthesize(
        prepare, classical_inputs={"amplitudes": data}, max_qubits=n, max_workspace=0
    )
    assert program.registers["q"] == tuple(range(n))
    return fq.Circuit.from_ir(program.circuit, dtype=COMPLEX)

In [ ]:
normalized_heat = heat / heat.norm()
heat_program = synth.synthesize(
    prepare, classical_inputs={"amplitudes": tuple(normalized_heat.tolist())},
    max_qubits=2, max_workspace=0,
)
print("Synthesized heat preparation registers:", heat_program.registers)
print("Preparation resources:", estimate_resources(heat_program.circuit))
heat_result = fq.run(heat_program)
print("Prepared amplitudes:", heat_result.statevector())

## 3. Construct the thermal block encoding

The seven-wire circuit uses three selector wires, two block wires and two data wires. `compose`, `adjoint` and `any(matrix=...)` assemble the LCU encoding. The top-left block should equal `A / alpha`; we execute all basis columns to check it. Explicit dense `any` operators are simulator-level operations, not a decomposed hardware or Clifford+T resource count.

In [ ]:
THERMAL_DATA, THERMAL_BLOCK, THERMAL_WIDTH = 2, 5, 7
FULL_QAE_BITS, MOCK_QAE_BITS = 5, 1
QAE_PHASE_WEIGHT = math.sin(math.pi / (1 << FULL_QAE_BITS))
DESIGN_BITS = len(EDGES)
def pair_block(i: int, j: int, *, enabled: bool) -> torch.Tensor:
    """Encode the thermal pair on flag, pair-ancilla and two data bits."""
    condition = i ^ j
    transform = torch.zeros(16, 16, dtype=COMPLEX)
    if not enabled:
        for word in range(16):
            transform[word ^ 8, word] = 1
        return transform
    for word in range(16):
        point = word & 3
        linked = point in (i, j)
        rotate = (word & 4) != 0
        target = word ^ (0 if linked else 8)
        if rotate:
            target ^= condition
        transform[target, word] = -1 if rotate else 1
    h = torch.tensor(((1, 1), (1, -1)), dtype=COMPLEX) / math.sqrt(2)
    lift = torch.kron(
        torch.eye(2, dtype=COMPLEX), torch.kron(h, torch.eye(4, dtype=COMPLEX))
    )
    return lift @ transform @ lift


def thermal_encoding(word: int = 63) -> fq.Circuit:
    """Use the source's six pair terms, ambient identity and unused selector."""
    prep = preparation((torch.tensor(weights(), dtype=REAL) / (alpha() / 2)).sqrt())
    selection = torch.eye(128, dtype=COMPLEX)
    for index, (i, j) in enumerate(EDGES):
        portion = slice(16 * index, 16 * (index + 1))
        selected = bool(word & (1 << (DESIGN_BITS - 1 - index)))
        selection[portion, portion] = pair_block(i, j, enabled=selected)
    circuit = fq.Circuit(THERMAL_WIDTH, dtype=COMPLEX).compose(prep, qubits=(0, 1, 2))
    circuit.any(*range(THERMAL_WIDTH), matrix=selection)
    return circuit.compose(prep.adjoint(), qubits=(0, 1, 2))


def unitary(circuit: fq.Circuit) -> torch.Tensor:
    """Read a small local unitary by executing each basis column."""
    dimension = 1 << circuit.n_qubits
    if dimension > 128:
        raise ValueError("The verification batch is limited to seven wires")
    batched = fq.Circuit.from_ir(
        circuit.to_ir(),
        dtype=COMPLEX,
        bsz=dimension,
        inputs=torch.eye(dimension, dtype=COMPLEX),
    )
    state: torch.Tensor = batched.state().reshape(dimension, dimension)
    return state.T.contiguous()

In [ ]:
block = thermal_encoding(63)
signal = unitary(block)
matrix_error = float((signal[:4, :4] * alpha() - matrix).abs().max())
assert matrix_error < 1e-10
print("Block-encoding maximum error:", matrix_error)
print("Block resources:", estimate_resources(block))

## 4. Check every coherent design branch

Here the six design wires control the thermal oracle. All 64 normalized heat inputs are executed in one batch using `Circuit.from_ir(..., inputs=..., bsz=64)`. The check compares each projected branch against its independently regenerated classical matrix and rejects leakage between design words.

In [ ]:
def conditional_thermal_encoding() -> fq.Circuit:
    """Define the original conditional thermal oracle for every six-bit design."""
    prep = preparation((torch.tensor(weights(), dtype=REAL) / (alpha() / 2)).sqrt())
    selectors = (6, 7, 8)
    thermal_wires = tuple(range(6, 13))
    circuit = fq.Circuit(13, dtype=COMPLEX).compose(prep, qubits=selectors)
    for index, (i, j) in enumerate(EDGES):
        operator = torch.eye(256, dtype=COMPLEX)
        for active in (False, True):
            offset = 128 * int(active) + 16 * index
            span = slice(offset, offset + 16)
            operator[span, span] = pair_block(i, j, enabled=active)
        circuit.any(index, *thermal_wires, matrix=operator)
    return circuit.compose(prep.adjoint(), qubits=selectors)


def verify_conditional(heat: torch.Tensor) -> tuple[float, ResourceEstimate]:
    circuit = conditional_thermal_encoding()
    dimension = 1 << circuit.n_qubits
    initial = torch.zeros(CONFIGURATIONS, dimension, dtype=COMPLEX)
    for word in range(CONFIGURATIONS):
        initial[word, word * 128 : word * 128 + 4] = heat
    execution = fq.Circuit.from_ir(
        circuit.to_ir(), dtype=COMPLEX, inputs=initial, bsz=CONFIGURATIONS
    )
    states: torch.Tensor = execution.state().reshape(
        CONFIGURATIONS, CONFIGURATIONS, 128
    )
    values = torch.arange(CONFIGURATIONS)
    diagonal = states[values, values]
    reference = torch.stack(
        [
            build_matrix(word).to(COMPLEX) @ heat / alpha()
            for word in range(CONFIGURATIONS)
        ]
    )
    maximum = float((diagonal[:, :4] - reference).abs().max())
    leakage = float((states.abs().square().sum() - diagonal.abs().square().sum()).abs())
    if maximum > 1e-10 or leakage > 1e-10:
        raise ValueError(
            "The coherent design register or its conditioned thermal projection changed"
        )
    return maximum, estimate_resources(circuit)

In [ ]:
conditional_error, conditional_resources = verify_conditional(normalized_heat.to(COMPLEX))
print("64-branch conditional maximum error:", conditional_error)
print("Conditional resources:", conditional_resources)

## 5. Fit the reciprocal and build the inverse circuit

The reciprocal is a classical polynomial/phase-fitting step. The **thermal** interval starts at `1 / (alpha · R_ENV)`. A separate illustration uses `[1/3, 1]` and target `0.3/x`; these targets must not be confused.

The visible inverse circuit prepares signed heat amplitudes, applies twelve phase rotations and eleven alternating block/adjoint queries, then projects the signal/work wires. No opaque solver call replaces this construction.

In [ ]:
@dataclass(frozen=True)
class Reciprocal:
    coefficients: torch.Tensor
    scale: float
    fit: PhaseFit
    minimum: float


def inverse_target(
    minimum: float, degree: int, *, target_scale: float | None = None
) -> Reciprocal:
    """Fit the odd mapped Chebyshev reciprocal at the source degree."""
    order = (degree + 1) // 2
    shift = 1 + minimum**2
    span = 1 - minimum**2
    x = (math.pi * (torch.arange(degree + 1, dtype=REAL) + 0.5) / (degree + 1)).cos()

    def recurrence(mapped: torch.Tensor) -> torch.Tensor:
        previous, value = torch.ones_like(mapped), mapped
        for _ in range(1, order):
            previous, value = value, 2 * mapped * value - previous
        return value

    normalizer = recurrence(torch.tensor(-shift / span, dtype=REAL))
    y = (1 - recurrence((2 * x.square() - shift) / span) / normalizer) / x
    degrees = torch.arange(1, degree + 1, 2, dtype=REAL)
    coefficients = (2 / x.numel()) * torch.cos(torch.acos(x)[:, None] * degrees).T @ y
    dense = torch.linspace(0, math.pi, 16385, dtype=REAL)
    max_p = float((torch.cos(dense[:, None] * degrees) @ coefficients).abs().max())
    amplitude_scale = 0.95 / max_p if target_scale is None else target_scale
    if amplitude_scale * max_p >= 1:
        raise ValueError("The scaled reciprocal must remain bounded")
    coefficients *= amplitude_scale
    return Reciprocal(coefficients, amplitude_scale, fit(coefficients), minimum)


def spectral_solution(
    matrix: torch.Tensor, reciprocal: Reciprocal, rhs: torch.Tensor
) -> torch.Tensor:
    values, directions = torch.linalg.eigh(matrix / alpha())
    bounded = response(values, reciprocal.fit.phases)
    result: torch.Tensor = directions @ (bounded * (directions.T @ rhs))
    return result


def inverse_circuit(reciprocal: Reciprocal) -> fq.Circuit:
    """Prepare the signed heat vector and apply all eleven encoded inverse calls."""
    block = thermal_encoding()
    inverse = block.adjoint()
    angles = reciprocal.fit.phases - math.pi / 2
    angles[0] += math.pi / 4
    angles[-1] += math.pi / 4
    if angles.numel() != 12:
        raise ValueError("Expected the source degree-11 reciprocal")
    heat = torch.tensor(HEAT, dtype=REAL)
    circuit = (
        fq.Circuit(8, dtype=COMPLEX)
        .compose(preparation(heat / heat.norm()), qubits=(6, 7))
        .h(0)
    )
    words = torch.arange(64)
    signs = 2 * (((words & 31) == 0) == (words >> 5 == 0)).to(REAL) - 1
    for index, angle in enumerate(angles):
        circuit.any(*range(6), matrix=torch.diag((1j * angle * signs).exp()))
        if index != angles.numel() - 1:
            circuit.compose(
                inverse if index % 2 == 0 else block, qubits=tuple(range(1, 8))
            )
    return circuit.h(0)

In [ ]:
reciprocal = inverse_target(1 / (alpha() * R_ENV), 11)
demo_reciprocal = inverse_target(1 / 3, 11, target_scale=0.3)
grid = torch.linspace(1 / 3, 1, 1025, dtype=REAL)
assert float((response(grid, demo_reciprocal.fit.phases) - 0.3 / grid).abs().max()) < 0.1
inverse = inverse_circuit(reciprocal)
print("Thermal inverse phase count:", reciprocal.fit.phases.numel())
print("Thermal inverse resources:", estimate_resources(inverse))

## 6. Execute and verify the thermal solution

`fq.run()` executes the inverse; extracting its first four amplitudes is a projection, not a deterministic hardware readout. Rescaling gives approximate signed temperature offsets. Compare the state with independent diagonalization and the physical solution with `torch.linalg.solve`. The degree-11 approximation allows less than 4% relative thermal error.

In [ ]:
inverse_result = fq.run(inverse)
projected = -inverse_result.statevector().flatten()[:4]
independent = spectral_solution(matrix, reciprocal, normalized_heat)
inverse_error = float((projected - independent).abs().max())
computed = projected.real * heat.norm() / (alpha() * reciprocal.scale)
relative_error = float((computed - reference).norm() / reference.norm())
assert inverse_error < 1e-10
assert relative_error < 0.04
print("Spectral amplitude error:", inverse_error)
print("Relative thermal error:", relative_error)
print("QSVT signed offsets (K):", computed.tolist())

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
axes[0].plot(grid.tolist(), (0.3 / grid).tolist(), "--", label="0.3/x")
axes[0].plot(grid.tolist(), response(grid, demo_reciprocal.fit.phases).tolist(), label="Degree-11 illustration")
axes[0].set(xlabel="Encoded value", ylabel="Response", title="Separate reciprocal demonstration")
axes[0].legend()
nodes = torch.arange(1, 5, dtype=REAL)
axes[1].bar((nodes - 0.18).tolist(), reference.tolist(), width=0.35, label="Classical solve")
axes[1].bar((nodes + 0.18).tolist(), computed.tolist(), width=0.35, label="QSVT")
axes[1].set(xlabel="Thermal node", ylabel="Offset (K)", title="Thermal result comparison")
axes[1].legend()
plt.show()
plt.close(figure)

## 7. Define the separate one-phase-bit mock design circuit

The following **64 supplied mock costs** are data, not quantum-computed thermal costs. `qae_state` constructs a one-bit mock phase-estimation state classically; high-level synthesis prepares its amplitudes. `mock_qaoa` builds a reusable eight-wire symbolic circuit with six design wires, one phase wire and one acceptance wire. This does not execute the full five-bit thermal QAE algorithm.

In [ ]:
MOCK_COSTS = [0.7142857142857143, 0.7142857142857143, 0.7142857142857143, 0.7142857142857143, 0.7142857142857143, 0.7142857142857143, 0.7142857142857143, 0.7142857142857143, 0.16483516483516486, 0.17970882620564155, 0.4295206454674894, 0.3881254478775872, 0.16483516483516486, 0.28688850853799314, 0.37006424072485306, 0.3851575130459745, 0.4120879120879121, 0.2616014558689719, 0.4120879120879121, 0.38361262588066714, 0.6110583768391079, 0.4526198204274334, 0.49200652869907313, 0.4482818851196006, 0.0992063492063492, 0.09920634920634923, 0.30111179740580607, 0.30209284943111037, 0.2713315097502867, 0.28607614460118413, 0.3449675324675325, 0.34496753246753253, 1.0, 1.0, 0.6770186335403726, 0.6649159663865546, 0.7888198757763975, 0.7027310924369746, 0.5763546798029556, 0.5763546798029556, 0.5324675324675325, 0.5166112956810632, 0.5225806451612903, 0.4814162914449447, 0.41457725947521856, 0.4381744146589753, 0.42702284165698784, 0.4334742622771895, 0.7077922077922081, 0.5764119601328905, 0.49067055393586, 0.47360027146250433, 0.700921658986175, 0.5447810069586574, 0.4961285327138985, 0.470387688631285, 0.403061224489796, 0.403061224489796, 0.39534673184863306, 0.3954478487776864, 0.3977910555857325, 0.3967962643893426, 0.39256440281030447, 0.39256440281030447]
costs = torch.tensor(MOCK_COSTS, dtype=REAL)
assert costs.shape == (CONFIGURATIONS,)
assert torch.isfinite(costs).all() and ((costs >= 0) & (costs <= 1)).all()
def qae_state(cost: float) -> torch.Tensor:
    """Form the source's two signed eigenphase branches for one phase bit."""
    phase = math.asin(cost)
    k = torch.arange(2, dtype=REAL)
    counter = torch.arange(2, dtype=REAL)[:, None]
    positive = (1j * k * (2 * phase - math.pi * counter)).exp().mean(dim=1)
    negative = (1j * k * (-2 * phase - math.pi * counter)).exp().mean(dim=1)
    cp = (math.sin(phase) - 1j * math.cos(phase)) / math.sqrt(2)
    cm = (math.sin(phase) + 1j * math.cos(phase)) / math.sqrt(2)
    accepted = (cp * positive + cm * negative) / math.sqrt(2)
    rejected = 1j * (cp * positive - cm * negative) / math.sqrt(2)
    amplitudes = torch.stack((rejected, accepted), dim=1).flatten()
    rounded = torch.complex(
        amplitudes.real.round(decimals=10), amplitudes.imag.round(decimals=10)
    )
    normalized: torch.Tensor = rounded / rounded.norm()
    return normalized


def complex_preparation(amplitudes: torch.Tensor) -> fq.Circuit:
    """Prepare passed complex amplitudes with real magnitude rotations and phases."""
    real = preparation(amplitudes.abs())
    circuit = fq.Circuit(2, dtype=COMPLEX).compose(real)
    return circuit.any(0, 1, matrix=torch.diag((1j * amplitudes.angle()).exp()))


def mock_qaoa(costs: torch.Tensor) -> fq.Circuit:
    """Build and reuse the same symbolic one-layer eight-qubit mock """
    selection = torch.zeros((256, 256), dtype=COMPLEX)
    for index in range(CONFIGURATIONS):
        prep = complex_preparation(qae_state(float(costs[index])))
        span = slice(4 * index, 4 * (index + 1))
        selection[span, span] = unitary(prep)
    wires = tuple(range(8))
    circuit = fq.Circuit(8, dtype=COMPLEX)
    for wire in range(DESIGN_BITS):
        circuit.h(wire)
    circuit.any(*wires, matrix=selection)
    circuit.phase(6, theta=QAE_PHASE_WEIGHT * fq.Parameter("gamma"))
    circuit.any(*wires, matrix=selection.mH)
    beta = fq.Parameter("beta")
    for wire in range(DESIGN_BITS):
        circuit.rx(wire, theta=-2 * beta)
    return circuit

## 8. Bind two angle pairs, run and compare

`fq.Parameter`, `bind_parameters` and `fq.run` reuse the same circuit topology. An independent tensor calculation below checks the complete complex state, rather than only a histogram. Exact probabilities are simulator results, not finite-shot QPU measurements. The reported mean is the mean **mock input cost**, not a physical cooling objective; two trials are not an optimizer convergence claim.

In [ ]:
def qae_reference(cost: float) -> torch.Tensor:
    """One explicit controlled Grover step and inverse one-bit Fourier transform."""
    angle = math.asin(cost)
    b = math.cos(angle)
    initial = torch.tensor((b, cost), dtype=REAL)
    c, s = math.cos(2 * angle), math.sin(2 * angle)
    rotate = torch.tensor(((c, -s), (s, c)), dtype=REAL)
    rotated = rotate @ initial
    vector = torch.cat((initial + rotated, initial - rotated)) / 2
    normalized: torch.Tensor = vector / vector.norm()
    return normalized.to(COMPLEX)


def check_qae(costs: torch.Tensor) -> None:
    for value in costs:
        cost = float(value)
        actual = qae_state(cost)
        expected = qae_reference(cost)
        torch.testing.assert_close(actual, expected, atol=1e-10, rtol=0)


def preparation_reference(amplitudes: torch.Tensor) -> torch.Tensor:
    magnitudes = amplitudes.abs().reshape(2, 2)
    first, second = (float(row.norm()) for row in magnitudes)
    start = torch.tensor(((first, -second), (second, first)), dtype=COMPLEX)
    controlled = torch.zeros(4, 4, dtype=COMPLEX)
    for branch, row in enumerate(magnitudes):
        norm = float(row.norm())
        c, s = (float(row[0]) / norm, float(row[1]) / norm) if norm else (1.0, 0.0)
        part = slice(2 * branch, 2 * (branch + 1))
        controlled[part, part] = torch.tensor(((c, -s), (s, c)), dtype=COMPLEX)
    result: torch.Tensor = (1j * amplitudes.angle()).exp()[:, None] * (
        controlled @ torch.kron(start, torch.eye(2, dtype=COMPLEX))
    )
    return result


def mock_state(costs: torch.Tensor, gamma: float, beta: float) -> torch.Tensor:
    states = torch.empty(CONFIGURATIONS, 4, dtype=COMPLEX)
    diagonal = torch.ones(4, dtype=COMPLEX)
    diagonal[2:] = torch.exp(
        torch.tensor(1j * QAE_PHASE_WEIGHT * gamma, dtype=COMPLEX)
    )
    for index, value in enumerate(costs):
        vector = qae_state(float(value))
        states[index] = preparation_reference(vector).mH @ (diagonal * vector) / 8
    c, s = math.cos(beta), 1j * math.sin(beta)
    rotation = torch.tensor(((c, s), (s, c)), dtype=COMPLEX)
    mixer = rotation
    for _ in range(1, len(EDGES)):
        mixer = torch.kron(mixer, rotation)
    mixed: torch.Tensor = mixer @ states
    return mixed.flatten()

In [ ]:
check_qae(costs)
template = mock_qaoa(costs)
assert template.parameter_names == ("beta", "gamma")
BINDINGS = ((0.67, 0.31), (-0.41, 0.23))
trials = []
probability_rows = []
for gamma, beta in BINDINGS:
    bound = template.bind_parameters({"gamma": gamma, "beta": beta})
    trial_result = fq.run(bound)
    state = trial_result.statevector().flatten()
    expected = mock_state(costs, gamma, beta)
    state_error = float((state - expected).abs().max())
    assert state_error < 1e-10
    probabilities = state.reshape(CONFIGURATIONS, 4).abs().square().sum(dim=1)
    assert abs(float(probabilities.sum()) - 1) < 1e-10
    probability_rows.append(probabilities)
    trials.append({"gamma": gamma, "beta": beta, "state_error": state_error,
                   "mean_mock_cost": float(probabilities @ costs)})
print(json.dumps(trials, indent=2))
print("Mock circuit resources:", estimate_resources(template))
figure, axis = plt.subplots(figsize=(10, 3), layout="constrained")
for angles, probabilities in zip(BINDINGS, probability_rows):
    axis.plot(range(CONFIGURATIONS), probabilities.tolist(), label=str(angles))
axis.set(xlabel="Six-bit design word", ylabel="Exact probability", title="Two mock parameter trials")
axis.legend()
plt.show()
plt.close(figure)

## 9. Save and replay the actual circuit

Save the last mock execution with its validation report, alongside the thermal execution. Replay uses the saved executable circuit, not phase fitting or the full research workflow. A successful replay establishes artifact reproducibility for these executions; it does not validate quantum advantage or physical design optimality. Default saves never overwrite an existing artifact.

In [ ]:
report = {
    "schema": "flagqai.cooling-walkthrough.v1",
    "thermal": {"matrix_error": matrix_error, "conditional_error": conditional_error,
                "inverse_error": inverse_error, "relative_error": relative_error,
                "classical_offsets_kelvin": reference.tolist(), "qsvt_offsets_kelvin": computed.tolist()},
    "mock_trials": trials, "mock_costs_are_thermal_objectives": False,
    "full_20_qubit_model_executed": False, "quantum_advantage_claim": False,
}
recorded = replace(trial_result, metrics={**trial_result.metrics, "flagqai_cooling": report})
fq.save("cooling-mock.fqx", recorded.plan, result=recorded)
fq.save("cooling-thermal.fqx", inverse_result.plan, result=inverse_result)
archived = fq.load("cooling-mock.fqx")
replayed = fq.run(archived.program)
torch.testing.assert_close(replayed.statevector(), recorded.statevector(), atol=1e-10, rtol=0)
assert archived.result.metrics["flagqai_cooling"] == report
print(json.dumps(report, indent=2))
print("Saved mock circuit replay passed. Edit BINDINGS to explore another pair; choose new filenames to retain each version.")

## 10. Optional full model definition — no execution

The original full design uses `6 design + 5 phase + 9 solver = 20` wires. The definitions below retain the original controlled solver, QAE and QuSO construction, but **Run all does not call them**. Large controlled dense matrices make this a much more expensive simulator workload. Its existence is not evidence that the full design was run or optimized. Review resources and choose a suitable compute environment before calling `build_full_quso` explicitly.

In [ ]:
PHASE_BITS, SOLVER_BITS = 5, 9
WIDTH = len(EDGES) + PHASE_BITS + SOLVER_BITS
assert WIDTH == 20
def conditional_solver(reciprocal: Reciprocal) -> fq.Circuit:
    """Prepare the heat vector and encode its inverse for each design."""
    base = conditional_thermal_encoding()
    inverse = base.adjoint()
    circuit = fq.Circuit(15, dtype=COMPLEX)
    # The first solver bit is retained in the scratch layout. The native
    # real-response projector combines the two signal roles on the next wire.
    signal_wire = 7
    block_wires = tuple(range(8, 13))
    data = (13, 14)
    targets = (*range(6), *block_wires, *data)
    heat = torch.tensor(HEAT, dtype=REAL)
    circuit.compose(preparation(heat / heat.norm()), qubits=data)
    circuit.h(signal_wire)
    phases = reciprocal.fit.phases - math.pi / 2
    phases[0] += math.pi / 4
    phases[-1] += math.pi / 4
    words = torch.arange(64)
    signs = 2 * (((words & 31) == 0) == (words >> 5 == 0)).to(REAL) - 1
    for i, angle in enumerate(phases):
        projector = torch.diag((1j * angle * signs).exp())
        circuit.any(signal_wire, *block_wires, matrix=projector)
        if i < phases.numel() - 1:
            circuit.compose(inverse if i % 2 == 0 else base, qubits=targets)
    return circuit.h(signal_wire)


def controlled_step(step: fq.Circuit) -> fq.Circuit:
    """Add a phase-estimation control to the known local simulator operations."""
    result = fq.Circuit(step.n_qubits + 1, dtype=COMPLEX)
    control = step.n_qubits
    controlled_matrices: dict[
        tuple[int, tuple[int, ...], tuple[int, ...], bool], torch.Tensor
    ] = {}
    for item in step.to_ir().instructions:
        if item.name == "any":
            matrix = item.matrix
            if not isinstance(matrix, torch.Tensor) or matrix.ndim != 2:
                raise ValueError("Only explicit local matrices can be controlled")
            key = (
                matrix.data_ptr(),
                tuple(matrix.shape),
                matrix.stride(),
                matrix.is_conj(),
            )
            if key not in controlled_matrices:
                size = matrix.size(0)
                operator = torch.eye(2 * size, dtype=COMPLEX)
                operator[size:, size:] = matrix
                controlled_matrices[key] = operator
            result.any(control, *item.wires, matrix=controlled_matrices[key])
        else:
            width = len(item.wires)
            local = fq.Circuit(width, dtype=COMPLEX)
            local.gate(item.name, tuple(range(width)), params=dict(item.params))
            result.compose(local.control(1, width), qubits=(*item.wires, control))
    return result


def amplitude_estimation(reciprocal: Reciprocal) -> fq.Circuit:
    """Define the five-bit QAE stage used by the full QuSO """
    solver = conditional_solver(reciprocal)
    grover = fq.Circuit(solver.n_qubits, dtype=COMPLEX)
    good = torch.ones(128, dtype=COMPLEX)
    good[0] = -1
    grover.any(*range(len(EDGES), len(EDGES) + 7), matrix=torch.diag(good))
    grover.compose(solver.adjoint())
    zero = -torch.ones(512, dtype=COMPLEX)
    zero[0] = 1
    grover.any(*range(len(EDGES), solver.n_qubits), matrix=torch.diag(zero))
    grover.compose(solver)
    controlled = controlled_step(grover)

    design = tuple(range(len(EDGES)))
    phase = tuple(range(len(design), len(design) + PHASE_BITS))
    workspace = tuple(range(phase[-1] + 1, WIDTH))
    result = fq.Circuit(WIDTH, dtype=COMPLEX).compose(
        solver, qubits=(*design, *workspace)
    )
    for wire in phase:
        result.h(wire)
    for index, wire in enumerate(phase):
        for _ in range(1 << (PHASE_BITS - 1 - index)):
            result.compose(controlled, qubits=(*design, *workspace, wire))
    return result.compose(qft(PHASE_BITS, inverse=True), qubits=phase)


def build_full_quso(
    reciprocal: Reciprocal, gamma: float, beta: float
) -> fq.Circuit:
    """Return the full 20-bit design; the source leaves this run disabled."""
    if not all(map(math.isfinite, (gamma, beta))):
        raise ValueError("Full QuSO angles must be finite")
    oracle = amplitude_estimation(reciprocal)
    circuit = fq.Circuit(WIDTH, dtype=COMPLEX)
    for wire in range(len(EDGES)):
        circuit.h(wire)
    circuit.compose(oracle)
    phases = torch.arange(32, dtype=REAL) * math.pi / 32
    circuit.any(*range(6, 11), matrix=torch.diag((1j * gamma * phases.sin()).exp()))
    circuit.compose(oracle.adjoint())
    for wire in range(len(EDGES)):
        circuit.rx(wire, theta=beta)
    return circuit.compose(oracle)

## Sources and retained helpers

Expanded by the FlagQAI workbench from FlagQuantum revision `fb7c05184085f829f282780618014001a3da1714`. Original notebook SHA-256: `5fed8a8af740c90ea0c6a42238c12e8ef47399348fd103d37f17fb2c63a975c8`. Companion source and upstream notebook remain available; original licenses and attribution apply.

Core construction and execution are inline. The shared `fit` / `response` numerical phase solver remains imported, as do the FlagQuantum SDK, PyTorch and plotting libraries. You can inspect the companion phase solver through the source viewer. No teaching claim should be inferred merely from a printed stage label.